In [76]:
import pandas as pd

df = pd.read_csv('data/raw/day_approach_maskedID_timeseries.csv')
df = df.sort_values(['Athlete ID', 'Date'])
df['day'] = df.groupby('Athlete ID').cumcount()   # 0, 1, 2, ... within each athlete

feature_cols = ['total km', 'nr. sessions', 'km Z3-4', 'km Z5-T1-T2',
                'km sprinting', 'strength training', 'hours alternative',
                'perceived exertion', 'perceived trainingSuccess',
                'perceived recovery']
feature_cols = ['total km']

rolled = (df.groupby('Athlete ID')[feature_cols]
            .rolling(window=7)
            .agg(['mean', 'std', 'max'])
            .reset_index())

rolled.columns = ['_'.join(c).strip('_') for c in rolled.columns]
rolled = rolled.rename(columns={'level_1': 'orig_row'})   # row label in df
rolled = rolled.dropna()

rolled['start'] = df.loc[rolled['orig_row'], 'day'].to_numpy()
rolled['stop'] = rolled['start'] + 1
rolled['injury'] = df.loc[rolled['orig_row'], 'injury'].to_numpy()

rolled = rolled.drop(columns='orig_row')
#print(rolled.head())

       nr. sessions  total km  km Z3-4  km Z5-T1-T2  km sprinting  \
0               1.0       5.8      0.0          0.6           1.2   
1               0.0       0.0      0.0          0.0           0.0   
2               1.0       0.0      0.0          0.0           0.0   
3               0.0       0.0      0.0          0.0           0.0   
4               1.0       0.0      0.0          0.0           0.0   
...             ...       ...      ...          ...           ...   
42178           0.0       0.0      0.0          0.0           0.0   
42179           1.0       8.0      0.0          0.0           0.0   
42180           1.0      14.7      0.0          0.0           0.0   
42181           1.0      12.2      0.0          0.0           0.0   
42182           1.0       9.3      0.0          0.0           0.0   

       strength training  hours alternative  perceived exertion  \
0                    0.0               0.00                0.11   
1                    0.0             

GroupKFold prevents leakage
Stratification handles imbalance,
combine them.

groups = Athlete ID  keeps athletes intact
y = injury (row-level) balances rare event across folds
StratifiedGroupKFold combines both

In [75]:
from sklearn.model_selection import StratifiedGroupKFold
from lifelines import CoxTimeVaryingFitter
from lifelines.utils import concordance_index
import numpy as np

x = rolled.drop(columns=['Athlete ID', 'start', 'stop', 'injury'])

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True,random_state=1)

fold_c_indices = []

def risk_set_of_c_index(df):
    injuries = df[df['injury'] == 1]
    concordant, comparable = 0.0,0.0
    for idx, inj in injuries.iterrows():
        t = inj['stop']
        athlete_id = inj['Athlete ID'] #to exclude athlete from their own risk set
        risk_set = df[ (df['stop'] >= t) &
            (t > df['start']) & 
            (df['Athlete ID'] != athlete_id) &
            (~((df['injury'] == 1) & (df['stop'] == t) ) ) ]
        #compare inj vs risk_set hazards
  
        concordant += (inj['hazard'] > risk_set['hazard']).sum() + 0.5 * (inj['hazard'] == risk_set['hazard']).sum()# A is riskier than B
        comparable += len(risk_set)
    if comparable != 0.0:
        return concordant / comparable

for fold, (train_idx, test_idx) in enumerate(sgkf.split(x, rolled['injury'], rolled['Athlete ID'])):
    training_data = rolled.iloc[train_idx].copy()
    test_data = rolled.iloc[test_idx].copy()
    #print(training_data.head())
    ctv = CoxTimeVaryingFitter()
    ctv.fit(
        df=training_data,
        id_col="Athlete ID",
        event_col="injury",
        start_col="start",
        stop_col="stop",
    )
    training_data['hazard'] = ctv.predict_partial_hazard(training_data).values
    test_data['hazard'] = ctv.predict_partial_hazard(test_data).values
    train_c = risk_set_of_c_index(training_data.copy())
    test_c = risk_set_of_c_index(test_data.copy())
    fold_c_indices.append(test_c)
    print(f"Train C Index: {train_c}")
    print(f"Test C Index: {test_c}\n")
print(f"Test Average C Index: {np.mean(fold_c_indices):.4f} +/- {np.std(fold_c_indices, ddof=1):.4f}")


        
        

/Users/hayden/Documents/InjuryPredictor/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


ValueError: Cannot have number of splits n_splits=5 greater than the number of groups: 1.

In [73]:


import xgboost as xgb
from sklearn.metrics import precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score, average_precision_score, RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score, precision_recall_curve
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
auc = []
ap = []
for fold, (train_idx, test_idx) in enumerate(sgkf.split(x, rolled['injury'], rolled['Athlete ID'])):
    training_data = rolled.iloc[train_idx].copy()

    injury_labels = training_data['injury']
    training_data = training_data.drop(columns=['injury', 'Athlete ID','start','stop'])
    
    test_data = rolled.iloc[test_idx].copy()
    test_injury_labels = test_data['injury']
    test_data = test_data.drop(columns=['injury', 'Athlete ID','start','stop'])

    neg_count = np.sum(injury_labels == 0)
    pos_count = np.sum(injury_labels == 1)
    weight_ratio = neg_count / pos_count
    
    model = xgb.XGBClassifier(max_depth=2, n_estimators = 512, scale_pos_weight = weight_ratio, learning_rate=.01) 
    model.fit(training_data, injury_labels)
    injury_pred_proba = model.predict_proba(test_data)[:, 1]
    auc_score = roc_auc_score(test_injury_labels, injury_pred_proba)
    ap_score = average_precision_score(test_injury_labels, injury_pred_proba)
    auc.append(auc_score)
    ap.append(ap_score)
    print(f"Average Precision: {ap_score:.4f}")
    print(f"Test ROC AUC: {auc_score:.4f}")
    
print(f"Average Test ROC AUC: {np.mean(auc):.4f} +/- {np.std(auc, ddof=1):.4f}")
print(f"Average Test AP: {np.mean(ap):.4f} +/- {np.std(ap, ddof=1):.4f} ")
precision, recall, thresholds = precision_recall_curve(test_injury_labels, injury_pred_proba)
#RocCurveDisplay.from_estimator(model, test_data, test_injury_labels)
disp = PrecisionRecallDisplay(precision=precision, recall=recall, average_precision=ap_score)


/Users/hayden/Documents/InjuryPredictor/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


ValueError: Cannot have number of splits n_splits=5 greater than the number of groups: 1.